# 💳 Fraud Detection — SMOTE, Logistic Regression & Random Forest

**Oasis Infobyte Data Analytics — Level 2, Task 3**

This notebook implements the requested fraud-detection workflow using exploratory data analysis, a stratified hold-out test set, SMOTE on training data only, Logistic Regression, Random Forest, Precision, Recall, F1-score, ROC-AUC, confusion matrices, ROC analysis, feature importance and production scalability considerations. The raw `creditcard.csv` is expected at `../data/creditcard.csv` and is not committed because it exceeds GitHub's single-file limit.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from imblearn.over_sampling import SMOTE

DATA_PATH='../data/creditcard.csv'
RESULTS_DIR='../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
df=pd.read_csv(DATA_PATH)
print(df.shape)
print(df['Class'].value_counts())
print(f"Fraud percentage: {df['Class'].mean()*100:.4f}%")

## 📊 Exploratory Data Analysis

The dataset contains a highly imbalanced target. In addition to the class distribution, this section examines transaction amounts for fraudulent versus legitimate transactions and explores fraud activity by hour of day represented in the dataset.

In [ ]:
# Class distribution
class_counts=df['Class'].value_counts().rename(index={0:'Legitimate',1:'Fraud'})
plt.figure(figsize=(7,4.5))
sns.barplot(x=class_counts.index,y=class_counts.values)
plt.title('Transaction Class Distribution')
plt.xlabel('Transaction Type')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'class_distribution.png'),dpi=160)
plt.show()

In [ ]:
# Transaction amount distribution: legitimate vs fraudulent
amount_eda=df[['Amount','Class']].copy()
amount_eda['AmountLog']=np.log1p(amount_eda['Amount'])
plt.figure(figsize=(9,5))
sns.histplot(data=amount_eda,x='AmountLog',hue='Class',bins=50,element='step',stat='density',common_norm=False)
plt.title('Transaction Amount Distribution: Fraud vs Legitimate')
plt.xlabel('log(1 + Transaction Amount)')
plt.ylabel('Density')
plt.legend(title='Class',labels=['Fraud','Legitimate'])
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'amount_distribution.png'),dpi=160)
plt.show()

print('Median legitimate amount:', df.loc[df['Class']==0,'Amount'].median())
print('Median fraudulent amount:', df.loc[df['Class']==1,'Amount'].median())

### 🕐 Time-of-day analysis

The `Time` variable records elapsed seconds from the first transaction in the dataset rather than a calendar timestamp. Therefore, the hour below is a **relative/cyclical hour-of-day**, not a real-world clock time. This still allows us to examine whether the fraud rate changes across the 24-hour cycle represented by the dataset.

In [ ]:
time_eda=df[['Time','Class']].copy()
time_eda['Hour']=(time_eda['Time']/3600)%24
time_eda['Hour']=time_eda['Hour'].astype(int)
hourly_counts=time_eda.groupby(['Hour','Class']).size().unstack(fill_value=0)
hourly_counts=hourly_counts.reindex(columns=[0,1],fill_value=0)
hourly_counts.columns=['Legitimate','Fraud']
hourly_counts['Total']=hourly_counts['Legitimate']+hourly_counts['Fraud']
hourly_counts['FraudRatePct']=np.where(hourly_counts['Total']>0,hourly_counts['Fraud']/hourly_counts['Total']*100,0)

plt.figure(figsize=(10,5))
plt.plot(hourly_counts.index,hourly_counts['FraudRatePct'],marker='o')
plt.xticks(range(24))
plt.title('Fraud Rate by Relative Hour of Day')
plt.xlabel('Relative Hour')
plt.ylabel('Fraud Rate (%)')
plt.grid(alpha=.25)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'fraud_rate_by_hour.png'),dpi=160)
plt.show()

display(hourly_counts[['Legitimate','Fraud','FraudRatePct']].round(4))

## Why accuracy is not enough
Fraud is a rare class. A classifier can appear highly accurate while missing fraudulent transactions. For example, with fraud representing only about 0.17% of transactions, a model that predicted every transaction as legitimate would have very high accuracy but would detect **zero fraud cases**. Precision measures how many flagged transactions are actually fraud; Recall measures how much of the fraud is caught; F1 balances precision and recall; ROC-AUC measures ranking quality across thresholds.

In [ ]:
X=df.drop(columns='Class'); y=df['Class']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)

# Confirm that both train and test contain fraud cases and preserve the minority proportion.
print('Training class proportions:')
print(y_train.value_counts(normalize=True).rename({0:'Legitimate',1:'Fraud'}))
print('\nTest class proportions:')
print(y_test.value_counts(normalize=True).rename({0:'Legitimate',1:'Fraud'}))

# Scale for Logistic Regression. Random Forest is unaffected by monotonic scaling, so the same scaled matrix is used for a consistent pipeline.
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

# Apply SMOTE ONLY to training data. A 10% minority/majority ratio limits unnecessary synthetic samples.
smote=SMOTE(random_state=42,sampling_strategy=.10)
X_smote,y_smote=smote.fit_resample(X_train_s,y_train)
print('\nBefore SMOTE:',y_train.value_counts().to_dict())
print('After SMOTE:',y_smote.value_counts().to_dict())

In [ ]:
models={
 'Logistic Regression':LogisticRegression(max_iter=1000,random_state=42),
 'Random Forest':RandomForestClassifier(n_estimators=20,max_depth=16,min_samples_leaf=2,n_jobs=-1,random_state=42)
}
metrics=[]; predictions={}; probabilities={}
for name,model in models.items():
 model.fit(X_smote,y_smote)
 pred=model.predict(X_test_s); prob=model.predict_proba(X_test_s)[:,1]
 predictions[name]=pred; probabilities[name]=prob
 metrics.append({'Model':name,'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,prob)})
metrics_df=pd.DataFrame(metrics)
display(metrics_df.style.format({c:'{:.4f}' for c in ['Precision','Recall','F1','ROC-AUC']}))
metrics_df.to_csv(os.path.join(RESULTS_DIR,'model_metrics.csv'),index=False)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
for ax,(name,pred) in zip(axes,predictions.items()):
 sns.heatmap(confusion_matrix(y_test,pred),annot=True,fmt='d',cbar=False,ax=ax)
 ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'confusion_matrices.png'),dpi=160); plt.show()

plt.figure(figsize=(7,5))
for name,prob in probabilities.items():
 fpr,tpr,_=roc_curve(y_test,prob); plt.plot(fpr,tpr,label=f'{name} (AUC={roc_auc_score(y_test,prob):.4f})')
plt.plot([0,1],[0,1],'--',label='Random classifier')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve — SMOTE Models'); plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'roc_curve.png'),dpi=160); plt.show()

## 🎯 Which metric matters most for fraud detection?

**Recall is usually the most important starting metric when the cost of missing a fraudulent transaction is high.** High recall means the system catches a larger share of actual fraud. However, recall should not be maximised blindly: a model can achieve very high recall by flagging many legitimate transactions, which reduces precision and increases investigation workload.

The precision–recall trade-off should therefore be chosen using business costs. In this experiment, Logistic Regression has higher Recall (88.78%) but much lower Precision (35.22%), while Random Forest has 85.71% Recall and 80.00% Precision. For a practical fraud system, **Recall would typically be prioritised, with Precision and F1 used as guardrails**, and the final decision threshold should reflect the relative cost of missed fraud versus false alerts.

In [ ]:
rf=models['Random Forest']
importance=pd.DataFrame({'Feature':X.columns,'Importance':rf.feature_importances_}).sort_values('Importance',ascending=False)
importance.to_csv(os.path.join(RESULTS_DIR,'random_forest_feature_importance.csv'),index=False)
display(importance.head(15))
top=importance.head(15).sort_values('Importance')
plt.figure(figsize=(8,6)); plt.barh(top['Feature'],top['Importance']); plt.xlabel('Importance'); plt.title('Top 15 Fraud Features — Random Forest'); plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'feature_importance.png'),dpi=160); plt.show()

## 🚀 Scalability: 1 million transactions per hour

A requirement of **1,000,000 transactions per hour is approximately 277.8 transactions per second**. The model should therefore be designed as a stateless scoring service that can process transactions independently and scale horizontally.

A production architecture could use:
- **Streaming or micro-batched ingestion** so transactions are scored continuously rather than loading the entire dataset into memory.
- **Efficient feature generation** using vectorised operations, cached/reference features and a feature store where appropriate.
- **Parallel model serving** with multiple scoring workers behind a load balancer.
- **Autoscaling and headroom** so capacity can increase during traffic spikes. If one scoring worker reliably handles `N` transactions/second, the minimum number of workers is `ceil(277.8 / N)` before adding operational headroom.
- **Monitoring** for throughput, p95/p99 latency, fraud-rate changes, data-quality issues and model drift.
- **Threshold management and human review** so high-risk transactions can be routed for investigation without blocking every legitimate transaction.
- **Periodic retraining and validation** as fraud patterns change over time.

The exact infrastructure requirement cannot be claimed from this notebook alone: it must be established through load testing using the complete feature pipeline and the production serving environment.

## Conclusion
Random Forest achieved the strongest overall balance in the supplied run: Precision 0.8000, Recall 0.8571, F1 0.8276 and ROC-AUC 0.9778. Logistic Regression produced higher Recall (0.8878) but lower Precision (0.3522). The leading Random Forest features were V14, V17, V12, V10 and V3. These anonymised PCA-derived features should not be interpreted as business variables without further documentation.

For fraud detection, Recall is generally prioritised when missed fraud is more costly than false alarms, but Precision must remain high enough for the investigation process to be operationally manageable. The decision threshold should ultimately be selected using business costs.

For high-volume production scoring, a streaming/batched feature pipeline, parallel model serving, threshold optimisation, monitoring, load testing and retraining would be appropriate.